# 🚢 範例 02：鐵達尼號生還預測 (分類篇) — AI 協作實戰筆記本

本筆記本重現按照 `PROMPT_GUIDE.md` 向 AI 提問的成果。初學者可以觀察 AI 如何協助挑選特徵、辨識作弊欄位 (Data Leakage)、處理缺失值、切分測試集並解讀混淆矩陣。

## 階段 0：環境準備與中文字型設定

In [ ]:
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 設定繁體中文字型
try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "sans-serif"]
    plt.rcParams["axes.unicode_minus"] = False

print("環境與字型載入成功！")

## 階段 1：資料載入與探索性資料分析 (EDA)

根據 AI 的建議，我們排除無效序號 (`row.names`, `name`) 與未來的作弊欄位 (`boat`)，保留 `pclass`、`sex`、`age`、`embarked`。

In [ ]:
df = pd.read_csv("data/titanic.csv")
print(f"原始資料維度: {df.shape}")

# 挑選有效特徵
features = ["pclass", "sex", "age", "embarked"]
X = df[features]
y = df["survived"]

# 觀察男女生還率
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=df, x="sex", y="survived", ax=axes[0], palette="pastel")
axes[0].set_title("不同性別之生還率", fontsize=13)
axes[0].set_ylabel("生還率")

# 觀察不同艙等生還率
sns.barplot(data=df, x="pclass", y="survived", order=["1st", "2nd", "3rd"], ax=axes[1], palette="viridis")
axes[1].set_title("不同艙等之生還率", fontsize=13)
axes[1].set_ylabel("生還率")

plt.tight_layout()
plt.show()

## 階段 2：缺失值處理、編碼與分層切分 (Stratified Split)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 數值管線 (年齡補中位數 + 標準化)
num_cols = ["age"]
num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 類別管線 (補眾數 + One-Hot 編碼)
cat_cols = ["pclass", "sex", "embarked"]
cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# 整合預處理器
preprocessor = ColumnTransformer([
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols)
])

# 分層切分 (80% 訓練, 20% 測試)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"訓練集: {X_train.shape[0]} 筆 (生存比例: {y_train.mean():.2%})")
print(f"測試集: {X_test.shape[0]} 筆 (生存比例: {y_test.mean():.2%})")

## 階段 3：模型選型與交叉驗證 (邏輯迴歸 vs 隨機森林)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

pipe_lr = Pipeline([("prep", preprocessor), ("model", LogisticRegression(random_state=42))])
pipe_rf = Pipeline([("prep", preprocessor), ("model", RandomForestClassifier(n_estimators=100, random_state=42))])

cv_lr = cross_val_score(pipe_lr, X_train, y_train, cv=5, scoring="f1")
cv_rf = cross_val_score(pipe_rf, X_train, y_train, cv=5, scoring="f1")

print(f"[邏輯迴歸] 5折交叉驗證 F1-score: {cv_lr.mean():.4f} (± {cv_lr.std():.4f})")
print(f"[隨機森林] 5折交叉驗證 F1-score: {cv_rf.mean():.4f} (± {cv_rf.std():.4f})")

## 階段 4：測試集評估與混淆矩陣解讀

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pipe_lr.fit(X_train, y_train)
y_pred = pipe_lr.predict(X_test)

print("=== 邏輯迴歸測試集分類報告 ===")
print(classification_report(y_test, y_pred, target_names=["遇難 (0)", "生還 (1)"]))

# 繪製混淆矩陣
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["預測遇難 (0)", "預測生還 (1)"],
            yticklabels=["實際遇難 (0)", "實際生還 (1)"])
plt.title("測試集混淆矩陣 (Confusion Matrix)", fontsize=13)
plt.tight_layout()
plt.show()

## 階段 5：虛擬乘客生還機率推論試算 (Inference)

In [ ]:
def predict_passenger(pipeline, info):
    df_input = pd.DataFrame([info])
    prob = pipeline.predict_proba(df_input)[0][1]
    pred = pipeline.predict(df_input)[0]
    status = "生還 ✅" if pred == 1 else "遇難 ❌"
    return status, prob

passengers = {
    "Rose (頭等艙女性, 17歲)": {"pclass": "1st", "sex": "female", "age": 17.0, "embarked": "Southampton"},
    "Jack (三等艙男性, 20歲)": {"pclass": "3rd", "sex": "male", "age": 20.0, "embarked": "Southampton"},
    "小孩 (三等艙男童, 5歲)": {"pclass": "3rd", "sex": "male", "age": 5.0, "embarked": "Cherbourg"}
}

print("=== 虛擬乘客預測結果 ===")
for name, info in passengers.items():
    status, prob = predict_passenger(pipe_lr, info)
    print(f"{name:25s} -> 預測結果: {status} | 生還機率: {prob:.1%}")